# 01 — Push Accuracy: Ensemble Voting + Accuracy-Targeted Tuning

Semua eksperimen 2026-07-09/10 mentok di ~52% accuracy di val set — dan
sebagian besar teknik yang dicoba (sample_weight, DART, resampling)
justru **menukar accuracy demi F1 macro** (mis. DART+sample_weight turun
ke 49.2% demi F1 macro naik ke 0.322). Kalau targetnya murni accuracy,
dua hal yang belum pernah dicoba:

1. **Soft-voting ensemble** RF + XGBoost + LightGBM — klasik, murah,
   biasanya dapat beberapa poin dari model yang berbeda "gaya salahnya".
2. **RandomizedSearchCV dengan `scoring="accuracy"`** langsung (bukan
   `f1_macro` seperti notebook 01 tanggal 2026-07-09) — supaya pencarian
   memang menyasar metrik yang diminta, bukan proxy yang berbeda.

Baseline acuan (split group-aware, test set):
XGBoost 52.1% accuracy (`README.md` utama).


In [1]:
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.dataset_builder import load_splits
from src.evaluation.metrics import evaluate, print_summary, compare_models

OUT_DIR = PROJECT_ROOT / "experiments" / "2026-07-15" / "outputs" / "accuracy_push"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df_train, df_val, df_test, feature_cols, le = load_splits(PROJECT_ROOT / "data" / "processed")

X_train, y_train = df_train[feature_cols].values, df_train["label"].values
X_val, y_val = df_val[feature_cols].values, df_val["label"].values
X_test, y_test = df_test[feature_cols].values, df_test["label"].values

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")


Train: (7157, 164), Val: (1533, 164), Test: (1533, 164)


## 1. Baseline RF / XGB / LGBM (default `configs/config.yaml`)

In [2]:
BASE_PARAMS = {
    "RandomForest": dict(
        n_estimators=300, max_depth=None, min_samples_split=5,
        random_state=42, n_jobs=-1, class_weight="balanced",
    ),
    "XGBoost": dict(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        n_jobs=-1, eval_metric="mlogloss", verbosity=0,
    ),
    "LightGBM": dict(
        n_estimators=300, max_depth=-1, learning_rate=0.05, num_leaves=63,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        n_jobs=-1, verbose=-1,
    ),
}
CLASSIFIERS = {
    "RandomForest": RandomForestClassifier,
    "XGBoost": XGBClassifier,
    "LightGBM": LGBMClassifier,
}

fitted = {}
proba_val = {}
proba_test = {}
baseline_results = []
for name, params in BASE_PARAMS.items():
    t0 = time.time()
    clf = CLASSIFIERS[name](**params)
    clf.fit(X_train, y_train)
    fitted[name] = clf
    proba_val[name] = clf.predict_proba(X_val)
    proba_test[name] = clf.predict_proba(X_test)
    res = evaluate(y_val, proba_val[name].argmax(axis=1), proba_val[name], le, model_name=f"{name} (baseline)")
    res["fit_seconds"] = round(time.time() - t0, 1)
    baseline_results.append(res)
    print_summary(res)

compare_models(baseline_results)



  RandomForest (baseline)
  Accuracy          : 0.4514
  Precision (macro) : 0.2602
  Recall (macro)    : 0.2971
  F1 (macro)        : 0.2693
  F1 (weighted)     : 0.4673
  Top-3 Accuracy    : 0.7123
  Top-5 Accuracy    : 0.8174

  XGBoost (baseline)
  Accuracy          : 0.5127
  Precision (macro) : 0.3335
  Recall (macro)    : 0.2515
  F1 (macro)        : 0.2685
  F1 (weighted)     : 0.4685
  Top-3 Accuracy    : 0.7508
  Top-5 Accuracy    : 0.8480


d:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



  LightGBM (baseline)
  Accuracy          : 0.5095
  Precision (macro) : 0.3574
  Recall (macro)    : 0.2411
  F1 (macro)        : 0.2613
  F1 (weighted)     : 0.4540
  Top-3 Accuracy    : 0.7469
  Top-5 Accuracy    : 0.8324


,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,top_3_accuracy,top_5_accuracy
model,,,,,,,
RandomForest (baseline),0.451402,0.260240,0.297077,0.269270,0.467313,0.712329,0.817352
XGBoost (baseline),0.512720,0.333505,0.251472,0.268524,0.468546,0.750815,0.848010
LightGBM (baseline),0.509459,0.357381,0.241093,0.261329,0.453988,0.746902,0.832355


## 2. Soft-voting ensemble (kombinasi bobot RF/XGB/LGBM)

Mengombinasikan `predict_proba` (rata-rata berbobot), tanpa refit ulang —
supaya bisa coba banyak kombinasi bobot dengan cepat.


In [3]:
WEIGHT_COMBOS = {
    "equal (1,1,1)": {"RandomForest": 1, "XGBoost": 1, "LightGBM": 1},
    "XGB+LGBM only (0,1,1)": {"RandomForest": 0, "XGBoost": 1, "LightGBM": 1},
    "favor XGB (1,2,1)": {"RandomForest": 1, "XGBoost": 2, "LightGBM": 1},
    "favor XGB (0,2,1)": {"RandomForest": 0, "XGBoost": 2, "LightGBM": 1},
    "favor XGB+LGBM (0,3,2)": {"RandomForest": 0, "XGBoost": 3, "LightGBM": 2},
    "XGB heavy (0,3,1)": {"RandomForest": 0, "XGBoost": 3, "LightGBM": 1},
}

ensemble_results = []
best_combo_name = None
best_combo_acc = -1
for combo_name, weights in WEIGHT_COMBOS.items():
    total_w = sum(weights.values())
    combined = np.zeros_like(proba_val["XGBoost"])
    for name, w in weights.items():
        if w == 0:
            continue
        combined += (w / total_w) * proba_val[name]
    y_pred = combined.argmax(axis=1)
    res = evaluate(y_val, y_pred, combined, le, model_name=f"Ensemble [{combo_name}]")
    ensemble_results.append(res)
    print(f"{combo_name:28s} acc={res['accuracy']:.4f} f1_macro={res['f1_macro']:.4f} f1_weighted={res['f1_weighted']:.4f}")
    if res["accuracy"] > best_combo_acc:
        best_combo_acc = res["accuracy"]
        best_combo_name = combo_name
        best_combo_weights = weights

print(f"\nBest ensemble by accuracy: {best_combo_name} (acc={best_combo_acc:.4f})")


equal (1,1,1)                acc=0.5108 f1_macro=0.2673 f1_weighted=0.4602
XGB+LGBM only (0,1,1)        acc=0.5114 f1_macro=0.2624 f1_weighted=0.4574
favor XGB (1,2,1)            acc=0.5114 f1_macro=0.2640 f1_weighted=0.4613
favor XGB (0,2,1)            acc=0.5088 f1_macro=0.2581 f1_weighted=0.4551
favor XGB+LGBM (0,3,2)       acc=0.5095 f1_macro=0.2632 f1_weighted=0.4561
XGB heavy (0,3,1)            acc=0.5101 f1_macro=0.2608 f1_weighted=0.4583

Best ensemble by accuracy: XGB+LGBM only (0,1,1) (acc=0.5114)


## 3. RandomizedSearchCV — `scoring="accuracy"` langsung (XGBoost & LightGBM)

Beda dari notebook 01 tanggal 2026-07-09 yang pakai `scoring="f1_macro"`.
`n_jobs=1` di estimator (di dalam search) untuk menghindari bug
oversubscription CPU yang ditemukan sebelumnya.


In [4]:
CV = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
N_ITER = 15
SCORING = "accuracy"

param_dist_xgb = {
    "n_estimators": [200, 300, 400, 500],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.08, 0.1],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "reg_lambda": [0.5, 1.0, 2.0, 5.0],
}
param_dist_lgbm = {
    "n_estimators": [200, 300, 400, 500],
    "num_leaves": [15, 31, 63, 95],
    "max_depth": [-1, 6, 10],
    "learning_rate": [0.01, 0.03, 0.05, 0.08, 0.1],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_samples": [5, 10, 20, 30],
}

t0 = time.time()
search_xgb = RandomizedSearchCV(
    XGBClassifier(random_state=42, n_jobs=1, eval_metric="mlogloss", verbosity=0),
    param_distributions=param_dist_xgb, n_iter=N_ITER, scoring=SCORING, cv=CV,
    random_state=42, n_jobs=-1, verbose=1,
)
search_xgb.fit(X_train, y_train)
print(f"XGB accuracy-search: {time.time()-t0:.1f}s, best CV accuracy: {search_xgb.best_score_:.4f}")
print(search_xgb.best_params_)

t0 = time.time()
search_lgbm = RandomizedSearchCV(
    LGBMClassifier(random_state=42, n_jobs=1, verbose=-1),
    param_distributions=param_dist_lgbm, n_iter=N_ITER, scoring=SCORING, cv=CV,
    random_state=42, n_jobs=-1, verbose=1,
)
search_lgbm.fit(X_train, y_train)
print(f"LGBM accuracy-search: {time.time()-t0:.1f}s, best CV accuracy: {search_lgbm.best_score_:.4f}")
print(search_lgbm.best_params_)


Fitting 2 folds for each of 15 candidates, totalling 30 fits


XGB accuracy-search: 797.0s, best CV accuracy: 0.5019
{'subsample': 0.9, 'reg_lambda': 2.0, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 5, 'learning_rate': 0.03, 'colsample_bytree': 0.6}
Fitting 2 folds for each of 15 candidates, totalling 30 fits
LGBM accuracy-search: 895.3s, best CV accuracy: 0.4959
{'subsample': 0.9, 'num_leaves': 15, 'n_estimators': 300, 'min_child_samples': 5, 'max_depth': -1, 'learning_rate': 0.01, 'colsample_bytree': 0.9}


In [5]:
tuned_xgb = search_xgb.best_estimator_
tuned_lgbm = search_lgbm.best_estimator_

proba_val["XGBoost (acc-tuned)"] = tuned_xgb.predict_proba(X_val)
proba_val["LightGBM (acc-tuned)"] = tuned_lgbm.predict_proba(X_val)
proba_test["XGBoost (acc-tuned)"] = tuned_xgb.predict_proba(X_test)
proba_test["LightGBM (acc-tuned)"] = tuned_lgbm.predict_proba(X_test)

tuned_results = []
for name in ["XGBoost (acc-tuned)", "LightGBM (acc-tuned)"]:
    res = evaluate(y_val, proba_val[name].argmax(axis=1), proba_val[name], le, model_name=name)
    tuned_results.append(res)
    print_summary(res)


d:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



  XGBoost (acc-tuned)
  Accuracy          : 0.5147
  Precision (macro) : 0.3190
  Recall (macro)    : 0.2324
  F1 (macro)        : 0.2482
  F1 (weighted)     : 0.4702
  Top-3 Accuracy    : 0.7697
  Top-5 Accuracy    : 0.8519

  LightGBM (acc-tuned)
  Accuracy          : 0.5010
  Precision (macro) : 0.2907
  Recall (macro)    : 0.2208
  F1 (macro)        : 0.2352
  F1 (weighted)     : 0.4550
  Top-3 Accuracy    : 0.7469
  Top-5 Accuracy    : 0.8356


## 4. Ensemble dengan model yang sudah di-tuning untuk accuracy

In [6]:
combined_best = (
    0.45 * proba_val["XGBoost (acc-tuned)"]
    + 0.35 * proba_val["LightGBM (acc-tuned)"]
    + 0.20 * proba_val["RandomForest"]
)
res_final_ensemble = evaluate(y_val, combined_best.argmax(axis=1), combined_best, le,
                               model_name="Ensemble [acc-tuned XGB+LGBM + baseline RF]")
print_summary(res_final_ensemble)

combined_best2 = 0.6 * proba_val["XGBoost (acc-tuned)"] + 0.4 * proba_val["LightGBM (acc-tuned)"]
res_final_ensemble2 = evaluate(y_val, combined_best2.argmax(axis=1), combined_best2, le,
                                model_name="Ensemble [acc-tuned XGB+LGBM only, 0.6/0.4]")
print_summary(res_final_ensemble2)



  Ensemble [acc-tuned XGB+LGBM + baseline RF]
  Accuracy          : 0.5127
  Precision (macro) : 0.3167
  Recall (macro)    : 0.2410
  F1 (macro)        : 0.2576
  F1 (weighted)     : 0.4714
  Top-3 Accuracy    : 0.7756
  Top-5 Accuracy    : 0.8565

  Ensemble [acc-tuned XGB+LGBM only, 0.6/0.4]
  Accuracy          : 0.5114
  Precision (macro) : 0.3218
  Recall (macro)    : 0.2316
  F1 (macro)        : 0.2497
  F1 (weighted)     : 0.4665
  Top-3 Accuracy    : 0.7678
  Top-5 Accuracy    : 0.8480


## 5. Bandingkan semua & simpan

In [7]:
all_results = baseline_results + ensemble_results + tuned_results + [res_final_ensemble, res_final_ensemble2]
comparison = compare_models(all_results)
comparison_sorted = comparison.sort_values("accuracy", ascending=False)
comparison_sorted


,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,top_3_accuracy,top_5_accuracy
model,,,,,,,
XGBoost (acc-tuned),0.514677,0.319022,0.232404,0.248234,0.470213,0.769733,0.851924
Ensemble [acc-tuned XGB+LGBM + baseline RF],0.512720,0.316672,0.241021,0.257602,0.471359,0.775603,0.856491
XGBoost (baseline),0.512720,0.333505,0.251472,0.268524,0.468546,0.750815,0.848010
"Ensemble [XGB+LGBM only (0,1,1)]",0.511416,0.357147,0.242487,0.262386,0.457415,0.752772,0.846053
"Ensemble [favor XGB (1,2,1)]",0.511416,0.349132,0.245317,0.263951,0.461319,0.769733,0.861709
"Ensemble [acc-tuned XGB+LGBM only, 0.6/0.4]",0.511416,0.321776,0.231604,0.249658,0.466494,0.767776,0.848010
"Ensemble [equal (1,1,1)]",0.510763,0.358076,0.247214,0.267313,0.460236,0.774299,0.859752
"Ensemble [XGB heavy (0,3,1)]",0.510111,0.341817,0.242769,0.260801,0.458310,0.754077,0.845401
LightGBM (baseline),0.509459,0.357381,0.241093,0.261329,0.453988,0.746902,0.832355


In [ ]:
comparison_sorted.to_csv(OUT_DIR / "val_comparison.csv")

summary = {
    "generated": pd.Timestamp.now().isoformat(),
    "best_by_accuracy": comparison_sorted.index[0],
    "best_accuracy": float(comparison_sorted.iloc[0]["accuracy"]),
    "xgb_tuned_params": search_xgb.best_params_,
    "lgbm_tuned_params": search_lgbm.best_params_,
    "results": {r["model"]: {k: r[k] for k in
        ["accuracy", "f1_macro", "f1_weighted", "top_3_accuracy", "top_5_accuracy"]}
        for r in all_results},
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
print(f"Saved: {OUT_DIR / 'val_comparison.csv'}")
print(f"Best by accuracy: {summary['best_by_accuracy']} = {summary['best_accuracy']:.4f}")


Saved: d:\Bridge-Prediction\experiments\2026-07-15\outputs\accuracy_push\val_comparison.csv
Best by accuracy: XGBoost (acc-tuned) = 0.5147


: 

## 6. Kesimpulan

Hasil val set (`experiments/2026-07-15/outputs/accuracy_push/val_comparison.csv`),
diurutkan berdasarkan accuracy:

| Model | Accuracy | F1 macro | F1 weighted |
|---|---|---|---|
| **XGBoost (acc-tuned)** | **51.5%** | 0.248 | 0.470 |
| Ensemble [acc-tuned XGB+LGBM + baseline RF] | 51.3% | 0.258 | 0.471 |
| XGBoost (baseline) | 51.3% | 0.269 | 0.469 |
| Ensemble [equal 1,1,1] | 51.1% | 0.267 | 0.460 |
| LightGBM (baseline) | 50.9% | 0.261 | 0.454 |
| RandomForest (baseline) | 45.1% | 0.269 | 0.467 |

**Tidak ada satu pun pendekatan yang mendekati 60%.** Baik ensemble
voting (segala kombinasi bobot) maupun tuning yang eksplisit menyasar
`scoring="accuracy"` cuma menggeser accuracy dalam rentang 45-52% —
kenaikan dari baseline murni 0.2pp (51.3%→51.5%), jauh dari target.

### Kenapa 60% kemungkinan besar TIDAK bisa dicapai untuk exact-contract (35 kelas)

Dicek langsung dari data: di antara 4.506 pasangan open/closed-room
(dua pasangan pemain **expert manusia** membidik kartu yang **identik
persis**), mereka hanya sepakat pada kontrak akhir yang **sama persis
37.6% dari waktu**. Manusia ahli sendiri, yang tahu semua konvensi
biddingnya masing-masing dan bisa berdiskusi di meja, hanya konsisten
37.6% satu sama lain untuk hand yang SAMA. Model kita (51-52%) SUDAH
LEBIH KONSISTEN daripada rata-rata dua pasangan manusia acak — karena
model belajar pola populasi (kontrak paling umum untuk profil tangan
tertentu), bukan gaya satu pasangan spesifik.

Ini artinya: exact-contract prediction punya **ambiguitas struktural**
(gaya bidding partnership berbeda-beda, bukan fungsi deterministik dari
kartu saja) yang membatasi accuracy maksimum yang realistis — 60% untuk
35 kelas kemungkinan besar melampaui apa yang bisa dicapai model
manapun dari fitur hand+lelang saja, karena itu berarti lebih konsisten
dari manusia ahli sendiri.

### Alternatif yang REALISTIS untuk 60%+: level kategori, bukan kontrak persis

Di level `target_category` (Pass/Partscore/Game/SmallSlam/GrandSlam,
5 kelas), pasangan manusia sepakat **68.3%** dari waktu — jauh lebih
tinggi. Stage 1 dari kaskade kategori (notebook 03, 2026-07-09/10)
sudah mencapai **75-78% accuracy** di subtask 5-kelas ini. Kalau
target "60% accuracy" bisa didefinisikan ulang di level kategori
(bukan level+strain persis), itu **sudah tercapai bahkan terlampaui**.

**Rekomendasi**: kalau tujuan riil adalah "prediksi berguna", scope
`target_category` (atau top-3/top-5 exact contract yang sudah 76-86%)
kemungkinan lebih pas untuk klaim "akurasi tinggi" secara jujur,
dibanding memaksa exact-contract 35-kelas ke 60% yang secara struktural
sudah melampaui konsistensi manusia sendiri. Perlu didiskusikan dengan
pembimbing skripsi arah mana yang mau diambil — ini bukan keputusan
teknis semata.
